**Fig. 4D and Video S4: whole-tissue run on the maturing scale stage with the DSM pair**

Needs a GPU.

In [ ]:
# Working directory: data/fig4/ under the repository root, where the Fig. 4 deposit folder is downloaded
# (domain-specialized_finetuned_bootstrap/ = the DSM model pair, one folder per sample with its
# input volume and the final mask, wound_finetuning/).
# Set the EPICYCLE_ANALYSES_DATA environment variable to keep the deposit somewhere else.
import os, sys
from pathlib import Path
_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment.yml").exists() and (p / "data").is_dir()), None)
assert _root is not None, "start Jupyter from inside the EpiCycle-analyses repository"
os.chdir(Path(os.environ.get("EPICYCLE_ANALYSES_DATA", _root / "data")) / "fig4")
print("working directory:", os.getcwd())


# EpiCycle ScaledSegmentation Pipeline

Run 3D cell segmentation, reconstruction, and assembly on tiled image data.

**Usage:** Fill in the paths and parameters below, then Run All.

## 1. Paths & Input

In [ ]:
# ===== PATHS (edit these) =====

# Input Z-stack TIFF (the original microscope file name is in the README) — set this to tile from scratch, or leave None if already tiled
INPUT_TIF = "scale_maturing/input.tif"   # Set to None if tiles already exist

# Working directory (tiles + outputs go here)
# If INPUT_TIF is set, tiles will be created at WORK_DIR/tiles/
# If INPUT_TIF is None, WORK_DIR should contain existing tile subdirectories
WORK_DIR = "scale_maturing"

# CellPose model paths (.pth files)
LATERAL_MODEL = "domain-specialized_finetuned_bootstrap/models/domain-specialized_Bootstrap4Finetuned_lateral_500epoch_scratch_lenient_rescale_True"
AXIAL_MODEL = "domain-specialized_finetuned_bootstrap/models/domain-specialized_Bootrap4Finetuned_axial_500epoch_scratch_lenient_rescale_True"

# Output directory name (created inside WORK_DIR)
OUTPUT_NAME = "Maturing_Scale"

## 2. Resource Configuration

In [ ]:
# ===== RESOURCES (auto-detected from the job's allocation if None) =====

N_GPUS = None            # None = auto-detect from SLURM/CUDA
N_CPU_PER_WORKER = None  # None = auto (total_cores / n_gpus in parallel mode)

## 3. Stage Selection & Tiling Parameters

In [ ]:
# Which stages to run (set False to skip)
RUN_TILING = INPUT_TIF is not None   # Auto: tile if input TIFF provided
# RUN_TILING = False
RUN_SEGMENTATION = True
RUN_RECONSTRUCTION = True
RUN_ASSEMBLY = True 
RUN_GLOBAL_RECONSTRUCTION = True

# ===== TILING (Stage 0) =====
PATCH_SIZE = 1024           # Target XY patch size in pixels
# OVERLAP is shared with assembly (set in Section 6)
OVERLAP = 0.5

Z_THRESHOLD_METHOD = "hard"  # "hard", "percentile", or "default"
Z_THRESHOLD_VALUE = 0.07     # Intensity threshold for Z-range detection
CHANNEL_INDEX = 1            # Which channel for Z-range analysis
AUTO_CORRECT_PATCH = False   # Auto-find valid patch size if exact tiling fails

## 4. Segmentation Parameters

In [ ]:
# ===== SEGMENTATION (Stage 1) =====
# Recorded for reference: the stage is built without segment_params below, so the engine
# defaults were used (channels [2, 1], cell probability 0, flow threshold 0.2). The flow
# threshold acts on the 2D masks only; the 3D reconstruction uses the probability and flow maps.

# GPU allocation strategy: "auto", "fixed" (3-GPU triplets), "round_robin"
GPU_STRATEGY = "auto"

# CellPose parameters
FLOW_THRESHOLD = 0.4
CELLPROB_THRESHOLD = 0.0
CHANNELS = [2, 1]      # [signal, nucleus] — adjust for your data
BATCH_SIZE = 1024

## 5. Reconstruction Parameters

In [ ]:
# ===== RECONSTRUCTION (Stage 2) =====

# Per-tile direct reconstruction: the engine's u-Segment3D defaults, with gradient renormalisation
# off as in the engine version the runs used (EpiCycle v0.1.2 defaults to on).
RECON_RENORM_GRADIENT = False

# "sequential" (one tile at a time, all cores) or "parallel" (multiple tiles)
RECON_MODE = "parallel"
# RECON_MODE = "sequential"

# Max concurrent tiles in parallel mode (None = n_gpus)
# Rule of thumb: 1900 TB node / ~450 GB per 1920x1920 tile ≈ 4 max
RECON_MAX_WORKERS = None

# Post-processing
SAVE_PROB3D = True
SAVE_FLOW3D = True
SAVE_BINARY = True
BINARY_ERODE = True
BINARY_KERNEL_SIZE = 3
BINARY_BORDER_MARGIN = 2
BINARY_FILL_HOLES = True

# Delete stage1 intermediates after reconstruction to save disk
CLEANUP_INTERMEDIATES = False

## 6. Assembly Parameters

In [ ]:
# ===== ASSEMBLY (Stage 3) =====

# Full volume dimensions in pixels (H, W)
FULL_SIZE = (2560, 2560)

# Tile size and overlap (must match how tiles were cut)
TILE_SIZE = PATCH_SIZE

# Number of Z slices (None = auto-detect from first tile)
DEPTH = None

# Which volumes to stitch
STITCH_MASK = True
STITCH_PROB = True
STITCH_BINARY = True

# Overlap blending for probabilities: "average" or "max"
PROB_STITCH_MODE = "max"

## 7. Global Indirect Reconstruction Parameters

In [ ]:
# ===== GLOBAL INDIRECT RECONSTRUCTION (Stage 4) =====

DOWNSAMPLE_FACTOR = 2          # Downsample stitched volume by this factor
GLOBAL_MIN_SIZE = 1000         # Remove objects smaller than this (voxels, after downsampling)
USE_PRECOMPUTED_BINARY = True  # Use stitched binary mask as foreground constraint

# Gradient descent parameters
# GLOBAL_N_ITER = 30            # Gradient descent iterations (more = finer, slower)
GLOBAL_N_ITER = 300            # Gradient descent iterations (more = finer, slower)
GLOBAL_MOMENTA = 0.9          # Momentum for gradient descent
# GLOBAL_GRADIENT_DECAY = 0.001    # Gradient decay (lower = more oversegmentation) -- since it is downsampled, go lower
# GLOBAL_GRADIENT_DECAY = 0.01    # Gradient decay (lower = more oversegmentation) -- since it is downsampled, go lower
# GLOBAL_GRADIENT_DECAY = 0.1    # Gradient decay (lower = more oversegmentation) -- since it is downsampled, go lower
# GLOBAL_GRADIENT_DECAY = 0.05    # Gradient decay (lower = more oversegmentation) -- since it is downsampled, go lower
GLOBAL_GRADIENT_DECAY = 0.25    # Gradient decay (lower = more oversegmentation) -- since it is downsampled, go lower

GLOBAL_RENORM_GRADIENT = True  # Renormalize gradient at each step

# Indirect method parameters
# GLOBAL_DTFORM_METHOD = "edt"   # Distance transform method edt is simple but there maybe artifacts in the seems
GLOBAL_DTFORM_METHOD = "edt"   # Distance transform method edt is simple but there maybe artifacts in the seems
GLOBAL_SMOOTH_SKEL_SIGMA = 1   # Skeleton smoothing sigma
GLOBAL_N_CPU = 64              # CPU threads for multiprocess watershed
GLOBAL_TILE_SHAPE = None       # Tile shape for watershed (None = auto)

## 8. Setup & Pre-flight Check

In [ ]:
from pathlib import Path
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(levelname)s %(message)s")

# Resolve GPU count
from packages.epicycle_scaledseg.utils.resources import get_allocated_gpus
n_gpus = N_GPUS or get_allocated_gpus()

work_dir = Path(WORK_DIR)
work_dir.mkdir(parents=True, exist_ok=True)

# Output lives inside the working directory
output_dir = work_dir / OUTPUT_NAME
output_dir.mkdir(exist_ok=True)
log_dir = output_dir / "logs"
log_dir.mkdir(exist_ok=True)

# Tiles directory
tiles_dir = work_dir / "tiles"

if RUN_TILING:
    print(f"Input TIFF: {INPUT_TIF}")
    print(f"Tiles will be created in: {tiles_dir}")
    print(f"Patch size: {PATCH_SIZE}, overlap: {OVERLAP}")
else:
    # Discover existing tile directories
    tile_dirs = sorted([
        str(p) for p in work_dir.iterdir()
        if p.is_dir() and not p.name.startswith(("stage1", "stage2", "stage3", "stage4", "logs", ".", "epicycle"))
        and p.name != OUTPUT_NAME
    ])
    print(f"Found {len(tile_dirs)} existing tile directories in {work_dir}")
    for td in tile_dirs[:5]:
        print(f"  {Path(td).name}")
    if len(tile_dirs) > 5:
        print(f"  ... and {len(tile_dirs) - 5} more")

print(f"\nGPUs: {n_gpus}")
print(f"Output: {output_dir}")

Input TIFF: <run>/WT_Developing_Fish/8-15-25_1791-2012_cdh1GFP_actb-h2b-mSc_SL8.9_maturing scale - Stitched - Denoised.nd2-PROCESSED-ISO-2560x2560.tif
Tiles will be created in: <run>/WT_Developing_Fish/Maturing_Scale/tiles
Patch size: 1024, overlap: 0.5

GPUs: 3
Output: <run>/WT_Developing_Fish/Maturing_Scale/Maturing_Scale


## 9. Build & Run Pipeline

In [ ]:
from packages.epicycle_scaledseg.core.pipeline import EpiCyclePipeline

pipeline = EpiCyclePipeline(
    work_dir=str(work_dir),
    n_gpus=n_gpus,
    log_dir=str(log_dir),
)

# --- Stage 0: Tiling ---
if RUN_TILING:
    from packages.epicycle_scaledseg.core.tiling import TilingStage
    pipeline.add_stage(TilingStage(
        input_tif=INPUT_TIF,
        output_dir=str(tiles_dir),
        patch_size=PATCH_SIZE,
        overlap=OVERLAP,
        z_threshold_method=Z_THRESHOLD_METHOD,
        threshold_value=Z_THRESHOLD_VALUE,
        channel_index=CHANNEL_INDEX,
        auto_correct=AUTO_CORRECT_PATCH,
    ))
    # Tiling creates tile_dirs; use placeholder for now
    tile_dirs = [str(work_dir)]

# --- Stage 1: Segmentation ---
if RUN_SEGMENTATION:
    from packages.epicycle_scaledseg.core.segmentation import SegmentationStage
    pipeline.add_stage(SegmentationStage(
        lateral_model=LATERAL_MODEL,
        axial_model=AXIAL_MODEL,
        out_root=str(output_dir / "stage1_segmentation"),
        gpu_strategy=GPU_STRATEGY,
    ))

# --- Stage 2: Reconstruction ---
if RUN_RECONSTRUCTION:
    from packages.epicycle_scaledseg.core.reconstruction import ReconstructionStage
    from packages.epicycle_scaledseg.config.defaults import get_default_reconstruct_params
    reconstruct_params = get_default_reconstruct_params()
    reconstruct_params['gradient_descent']['renorm_gradient'] = RECON_RENORM_GRADIENT
    pipeline.add_stage(ReconstructionStage(
        seg_root=str(output_dir / "stage1_segmentation"),
        out_root=str(output_dir / "stage2_reconstruction"),
        mode=RECON_MODE,
        reconstruct_params=reconstruct_params,
        max_workers=RECON_MAX_WORKERS,
        n_cpu_per_worker=N_CPU_PER_WORKER,
        save_prob3d=SAVE_PROB3D,
        save_flow3d=SAVE_FLOW3D,
        save_binary=SAVE_BINARY,
        binary_erode=BINARY_ERODE,
        binary_kernel_size=BINARY_KERNEL_SIZE,
        binary_border_margin=BINARY_BORDER_MARGIN,
        binary_fill_holes=BINARY_FILL_HOLES,
        cleanup_intermediates=CLEANUP_INTERMEDIATES,
    ))

# --- Stage 3: Assembly ---
if RUN_ASSEMBLY:
    from packages.epicycle_scaledseg.core.assembly import AssemblyStage
    pipeline.add_stage(AssemblyStage(
        recon_root=str(output_dir / "stage2_reconstruction"),
        output_dir=str(output_dir / "stage3_assembly"),
        full_size=FULL_SIZE,
        tile_size=TILE_SIZE,
        overlap=OVERLAP,
        depth=DEPTH,
        stitch_mask=STITCH_MASK,
        stitch_prob=STITCH_PROB and SAVE_PROB3D,
        stitch_binary=STITCH_BINARY and SAVE_BINARY,
        prob_stitch_mode=PROB_STITCH_MODE,
    ))

# --- Stage 4: Global Indirect Reconstruction ---
if RUN_GLOBAL_RECONSTRUCTION:
    from packages.epicycle_scaledseg.core.global_reconstruction import GlobalReconstructionStage
    from packages.epicycle_scaledseg.config.defaults import get_default_indirect_reconstruct_params

    indirect_params = get_default_indirect_reconstruct_params(n_cpu=GLOBAL_N_CPU)
    indirect_params['gradient_descent']['n_iter'] = GLOBAL_N_ITER
    indirect_params['gradient_descent']['momenta'] = GLOBAL_MOMENTA
    indirect_params['gradient_descent']['gradient_decay'] = GLOBAL_GRADIENT_DECAY
    indirect_params['gradient_descent']['renorm_gradient'] = GLOBAL_RENORM_GRADIENT
    indirect_params['indirect_method']['dtform_method'] = GLOBAL_DTFORM_METHOD
    indirect_params['indirect_method']['smooth_skel_sigma'] = GLOBAL_SMOOTH_SKEL_SIGMA
    if GLOBAL_TILE_SHAPE is not None:
        indirect_params['gradient_descent']['tile_shape'] = GLOBAL_TILE_SHAPE

    pipeline.add_stage(GlobalReconstructionStage(
        assembly_dir=str(output_dir / "stage3_assembly"),
        output_dir=str(output_dir / "stage4_global"),
        downsample_factor=DOWNSAMPLE_FACTOR,
        min_size=GLOBAL_MIN_SIZE,
        use_precomputed_binary=USE_PRECOMPUTED_BINARY,
        indirect_params=indirect_params,
        n_cpu=GLOBAL_N_CPU,
        tile_shape=GLOBAL_TILE_SHAPE,
    ))

print(f"Pipeline: {len(pipeline.stages)} stages")
for s in pipeline.stages:
    print(f"  - {s.name}")
print(f"Tile dirs: {len(tile_dirs)}")
print(f"Output: {output_dir}")

# ===== RUN =====
result = pipeline.run(tile_dirs)
print()
print(result.summary())

2026-02-04 11:30:57,523 epicycle.pipeline INFO Pipeline starting: 5 stage(s), 1 tile(s), 3 GPU(s)
2026-02-04 11:30:57,524 epicycle.pipeline INFO Run parameters saved to data/WT_Developing_Fish/Maturing_Scale/Maturing_Scale/logs/run_params.json
2026-02-04 11:30:57,524 epicycle.pipeline INFO === Stage: tiling ===
2026-02-04 11:30:57,525 epicycle.tiling INFO Stage 'tiling' starting with 1 tile(s)
2026-02-04 11:30:57,525 epicycle.tiling INFO Parameters: {"input_tif": "data/WT_Developing_Fish/8-15-25_1791-2012_cdh1GFP_actb-h2b-mSc_SL8.9_maturing scale - Stitched - Denoised.nd2-PROCESSED-ISO-2560x2560.tif", "patch_size": 1024, "overlap": 0.5, "z_threshold_method": "hard", "threshold_value": 0.07, "channel_index": 1, "auto_correct": false}
2026-02-04 11:30:57,525 epicycle.core.tiling INFO Tiling data/WT_Developing_Fish/8-15-25_1791-2012_cdh1GFP_actb-h2b-mSc_SL8.9_maturing scale - Stitched - Denoised.nd2-PROCESSED-ISO-2560x2560.tif -> data/WT_Developing_Fish/Maturing_Scale/tiles (patch=1024, o

Pipeline: 5 stages
  - tiling
  - segmentation
  - reconstruction
  - assembly
  - global_reconstruction
Tile dirs: 1
Output: <run>/WT_Developing_Fish/Maturing_Scale/Maturing_Scale


2026-02-04 11:31:05,314 epicycle.utils.tiling INFO Z-stack shape: Z=599, C=2, H=2560, W=2560, dtype=uint16
2026-02-04 11:31:05,315 epicycle.utils.tiling INFO Tiling: patch_size=1024, step=512, overlap=0.5
2026-02-04 11:31:05,316 epicycle.utils.tiling INFO Grid: 4 rows x 4 cols = 16 tiles
Saving tiles: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 16/16 [00:20<00:00,  1.26s/tile]
2026-02-04 11:31:25,545 epicycle.core.tiling INFO Tiling complete: 16 tiles, actual patch=1024, step=512
2026-02-04 11:31:25,573 epicycle.pipeline INFO Stage 'tiling': 16/16 succeeded (28.0s)
2026-02-04 11:31:25,573 epicycle.tiling INFO Stage 'tiling' finished: 16/16 succeeded (28.0s)
2026-02-04 11:31:25,573 epicycle.pipeline INFO Stage 'tiling' produced 16 tile directories
2026-02-04 11:31:25,574 epicycle.pipeline INFO === Stage: segmentation ===
2026-02-04 11:31:25,575 epicycle.segmentation INFO Stage 'segmentation' starting with 16 til

Automatric binary thresholding:  0.6
[ 373 1216 1216] [(48, 48), (96, 96), (96, 96)]
using total number of tiles :  144
start dynamics
Tile id 15 0 0
Tile id 21 0 0
Tile id 17 0 0
Tile id 28 0 0
Tile id 27 0 0
Tile id 16 103 103
Tile id 26 0 0
Tile id 32 0 0
Tile id 22 354 354
Tile id 33 229 229
Tile id 10 913 913
Tile id 34 492 492
Tile id 31 1642 1642
Tile id 35 2529 2529
Tile id 4 12422 12422
Tile id 72 0 0
Tile id 5 7524 7524
Tile id 73 1 1
Tile id 11 7267 7267
Tile id 23 15072 15072
Tile id 29 20934 20934
Tile id 20 25833 25833
Tile id 30 39851 39851
Tile id 9 75018 75018
Tile id 78 0 0
Tile id 25 106306 106306
Tile id 3 203408 203408
Tile id 14 226327 226327
Tile id 84 13223 13223
Tile id 24 245941 245941
Tile id 59 271693 271693
Tile id 71 224809 224809
Tile id 79 127466 127466
Tile id 70 314848 314848
Tile id 19 422331 422331
Tile id 53 446833 446833
Tile id 65 367390 367390
Tile id 69 365845 365845
Tile id 74 350667 350667
Tile id 36 462846 462846
Tile id 68 403066 403066
Tile

Eroding (GPU): 100%|██████████| 738/738 [00:11<00:00, 62.24inst/s]  


Automatric binary thresholding:  0.6
[ 409 1216 1216] [(48, 48), (96, 96), (96, 96)]
using total number of tiles :  144
start dynamics
Tile id 35 0 0
Tile id 3 0 0
Tile id 2 0 0
Tile id 34 0 0
Tile id 29 7860 7860
Tile id 17 7211 7211
Tile id 22 15866 15866
Tile id 23 10831 10831
Tile id 25 17502 17502
Tile id 24 4084 4084
Tile id 32 19799 19799
Tile id 5 49691 49691
Tile id 33 23897 23897
Tile id 31 43277 43277
Tile id 7 43326 43326
Tile id 27 58120 58120
Tile id 30 40583 40583
Tile id 4 67326 67326
Tile id 13 52879 52879
Tile id 10 79721 79721
Tile id 11 67736 67736
Tile id 26 65409 65409
Tile id 28 63810 63810
Tile id 53 74135 74135
Tile id 70 0 0
Tile id 71 0 0
Tile id 16 112044 112044
Tile id 12 106931 106931
Tile id 21 109105 109105
Tile id 65 34807 34807
Tile id 19 129405 129405
Tile id 6 136580 136580
Tile id 1 125198 125198
Tile id 59 117385 117385
Tile id 63 63801 63801
Tile id 66 67701 67701
Tile id 64 84430 84430
Tile id 58 177455 177455
Tile id 47 173516 173516
Tile id 9 1

... (28 outputs left out)

Eroding (GPU): 100%|██████████| 865/865 [00:04<00:00, 181.64inst/s] 
2026-02-04 12:40:55,348 epicycle.pipeline INFO Stage 'reconstruction': 16/16 succeeded (2134.9s)
2026-02-04 12:40:55,349 epicycle.reconstruction INFO Stage 'reconstruction' finished: 16/16 succeeded (2134.9s)
2026-02-04 12:40:55,350 epicycle.pipeline INFO === Stage: assembly ===
2026-02-04 12:40:55,351 epicycle.assembly INFO Stage 'assembly' starting with 16 tile(s)
2026-02-04 12:40:55,351 epicycle.assembly INFO Parameters: {"full_size": [2560, 2560], "tile_size": 1024, "overlap": 0.5, "depth": null, "stitch_mask": true, "stitch_prob": true, "stitch_binary": true, "prob_stitch_mode": "max", "binary_format": "npy", "n_workers": 16}
2026-02-04 12:40:55,352 epicycle.assembly INFO Loading tiling config from data/WT_Developing_Fish/Maturing_Scale/tiles/tiling_config.json
2026-02-04 12:40:55,352 epicycle.assembly INFO Loading tiling config from data/WT_Developing_Fish/Maturing_Scale/tiles/tiling_config.json
2026-02-04 12:40

[ 395 1472 1472] [(48, 48), (96, 96), (96, 96)]
using total number of tiles :  256
start dynamics
Tile id 62 0 0
Tile id 58 0 0
Tile id 60 0 0
Tile id 63 0 0
Tile id 42 0 0
Tile id 50 0 0
Tile id 46 0 0
Tile id 54 0 0
Tile id 110 0 0
Tile id 38 0 0
Tile id 44 0 0
Tile id 52 0 0
Tile id 36 0 0
Tile id 118 0 0
Tile id 61 0 0
Tile id 59 0 0
Tile id 34 212 212
Tile id 126 0 0
Tile id 47 0 0
Tile id 39 0 0
Tile id 55 0 0
Tile id 111 0 0
Tile id 119 0 0
Tile id 127 0 0
Tile id 43 0 0
Tile id 51 0 0
Tile id 45 0 0
Tile id 53 0 0
Tile id 37 0 0
Tile id 130 0 0
Tile id 128 0 0
Tile id 136 0 0
Tile id 28 2511 2511
Tile id 35 212 212
Tile id 30 2842 2842
Tile id 14 3266 3266
Tile id 56 21621 21621
Tile id 6 4470 4470
Tile id 102 4565 4565
Tile id 144 0 0
Tile id 152 141 141
Tile id 131 0 0
Tile id 129 0 0
Tile id 26 13861 13861
Tile id 22 17819 17819
Tile id 31 0 0
Tile id 7 0 0
Tile id 20 34603 34603
Tile id 137 0 0
Tile id 40 40470 40470
Tile id 94 49814 49814
Tile id 4 45634 45634
Tile id 15 1

2026-02-04 12:54:48,759 epicycle.core.global_reconstruction INFO Global reconstruction complete: shape=(299, 1280, 1280), n_labels=3479, elapsed=660.4s
2026-02-04 12:54:48,761 epicycle.core.global_reconstruction INFO Saving outputs to <run>/WT_Developing_Fish/Maturing_Scale/Maturing_Scale/stage4_global
2026-02-04 12:55:34,480 epicycle.pipeline INFO Stage 'global_reconstruction': 1/1 succeeded (722.8s)
2026-02-04 12:55:34,481 epicycle.global_reconstruction INFO Stage 'global_reconstruction' finished: 1/1 succeeded (722.8s)
2026-02-04 12:55:34,482 epicycle.pipeline WARNING No tiles remaining — stopping pipeline.
2026-02-04 12:55:34,482 epicycle.pipeline INFO Pipeline: 1/16 tiles (5077.0s total)
  tiling: 16/16 succeeded (28.0s)
  segmentation: 16/16 succeeded (2034.9s)
  reconstruction: 16/16 succeeded (2134.9s)
  assembly: 16/16 succeeded (156.3s)
  global_reconstruction: 1/1 succeeded (722.8s)



Pipeline: 1/16 tiles (5077.0s total)
  tiling: 16/16 succeeded (28.0s)
  segmentation: 16/16 succeeded (2034.9s)
  reconstruction: 16/16 succeeded (2134.9s)
  assembly: 16/16 succeeded (156.3s)
  global_reconstruction: 1/1 succeeded (722.8s)


## 10. Check Status

In [ ]:
# Quick status check — how many tiles completed each stage?
seg_root = output_dir / "stage1_segmentation"
recon_root = output_dir / "stage2_reconstruction"

seg_done = sum(1 for td in tile_dirs if (seg_root / Path(td).name / "SEGMENTATION_DONE").exists())
recon_done = sum(1 for td in tile_dirs if (recon_root / Path(td).name / f"{Path(td).name}_mask.tif").exists())

print(f"Stage 1 (Segmentation):   {seg_done}/{len(tile_dirs)} complete")
print(f"Stage 2 (Reconstruction): {recon_done}/{len(tile_dirs)} complete")
print(f"Stage 3 (Assembly):       check {output_dir / 'stage3_assembly'}")
print(f"Stage 4 (Global Recon):   {'done' if (output_dir / 'stage4_global' / 'global_segmentation.tif').exists() else 'not done'}")

Stage 1 (Segmentation):   0/1 complete
Stage 2 (Reconstruction): 0/1 complete
Stage 3 (Assembly):       check <run>/WT_Developing_Fish/Maturing_Scale/Maturing_Scale/stage3_assembly
Stage 4 (Global Recon):   done
